### This notebook is for downloading and clipping raw data to nc
For downloading raw data, it queries APIs.  
For preprocessing, it currently clips to an NC mask (data/processed/nc_boundary.gpkg)


In [ ]:
import geopandas as gpd
import earthaccess
from collections import defaultdict
import geopandas as gpd
import rioxarray
from rioxarray.merge import merge_arrays
from peatfire import data_path
from peatfire.preproc import clip_vector_to_mask_and_save, clip_raster_to_mask_and_save, get_key
from pathlib import Path
import xarray as xr
import pandas as pd, requests, time
from datetime import date, timedelta
import rasterio
import io
import ee
import geemap
import os

### Raw data downloads

nc boundary download

In [15]:
url = 'https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_500k.zip'
states = gpd.read_file(url)            # geopandas reads the zip directly
nc = states[states['NAME'] == 'North Carolina']
nc.to_file('../data/processed/boundaries/nc_boundary.gpkg', driver='GPKG')   # GeoPackage > shapefile

earthaccess bulk download of MCD64A1

In [2]:
earthaccess.login()  # uses Earthdata credentials / .netrc

results = earthaccess.search_data(
    short_name='MCD64A1',
    version='061',
    temporal=('2000-11-01', '2026-06-03'), # set whatever dates you want - just do about a year for now for testing
    bounding_box=(-84.322, 33.842, -75.461, 36.588),  # NC bbox; returns h11v05 + h12v05
)
earthaccess.download(results, '../data/raw/fire/MCD64A1_061/')

/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)
QUEUEING TASKS | : 100%|██████████| 610/610 [00:00<00:00, 77745.46it/s]
PROCESSING TASKS | : 100%|██████████| 610/610 [01:51<00:00,  5.49it/s]
COLLECTING RESULTS | : 100%|██████████| 610/610 [00:00<00:00, 1685458.13it/s]


[PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2000306.h11v05.061.2021307220204.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2000306.h10v05.061.2021307220206.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2000336.h11v05.061.2021307220306.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2000336.h10v05.061.2021307220309.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001001.h10v05.061.2021307220407.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001001.h11v05.061.2021307220405.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001032.h10v05.061.2021307220531.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001032.h11v05.061.2021307220534.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001060.h11v05.061.2021307220624.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001060.h10v05.061.2021307220624.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001091.h10v05.061.2021307220732.hdf'),
 PosixPath

### Clipping to NC

first grab the NC bounds shapefile

In [7]:
nc = gpd.read_file(data_path('processed', 'boundaries', 'nc_boundary.gpkg'))

clip MCD64A1 to NC

In [8]:
def sds(hdf):
    """Return the GDAL subdataset string for the Burn Date layer of an MCD64A1 file."""
    with rasterio.open(str(hdf)) as src:
        for name in src.subdatasets:
            if name.rstrip().endswith("Burn Date"):
                return name
    raise ValueError(f"No 'Burn Date' subdataset found in {hdf.name}")

# 2. group the two tiles by acquisition date (filename token A2017001, A2017032, ...)
raw = data_path('raw', 'fire', 'MCD64A1_061')
by_date = defaultdict(list)
for f in sorted(raw.glob("MCD64A1.*.hdf")):
    by_date[f.name.split(".")[1]].append(f)

# 3. mosaic -> clip -> save, per date
out = data_path('processed', 'fire', 'MCD64A1_061')
out.mkdir(parents=True, exist_ok=True)

for date, files in by_date.items():
    tiles = [rioxarray.open_rasterio(sds(f), masked=True) for f in files]
    mosaic = merge_arrays(tiles) # stitch h10v05 + h11v05
    nc_sin = nc.to_crs(mosaic.rio.crs) # reproject NC -> sinusoidal
    clip = mosaic.rio.clip(nc_sin.geometry, nc_sin.crs, drop=True)
    clip.rio.to_raster(out / f"MCD64A1_{date}_nc.tif")
    

/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoref

clip VIIRS to NC

In [ ]:
viirs_snpp_archive_nc = clip_vector_to_mask_and_save(data_path('raw', 'fire', 'DL_FIRE_SV-C2_758389', 'fire_archive_SV-C2_758389.shp'), nc, data_path('processed', 'fire', 'viirs', f'viirs_snpp_archive_nc.gpkg'))
viirs_noaa20_nrt_nc = clip_vector_to_mask_and_save(data_path('raw', 'fire', 'DL_FIRE_J1V-C2_758431', 'fire_nrt_J1V-C2_758431.shp'), nc, data_path('processed', 'fire', 'viirs', f'viirs_noaa20_nrt_nc.gpkg'))
viirs_noaa21_nrt_nc = clip_vector_to_mask_and_save(data_path('raw', 'fire', 'DL_FIRE_J2V-C2_758399', 'fire_nrt_J2V-C2_758399.shp'), nc, data_path('processed', 'fire', 'viirs', f'viirs_noaa21_nrt_nc.gpkg'))

clip SE firemap to NC

In [ ]:
for year in range(2000, 2023):
    _ = clip_raster_to_mask_and_save(data_path('raw', 'fire', 'se_firemap', f'cbi_mosaic_{str(year)}', f'cbi_mosaic_{str(year)}.tif'), nc, data_path('processed', 'fire', 'se_firemap', f'cbi_mosaic_{str(year)}_nc', f'cbi_mosaic_{str(year)}_nc.tif'))

### Google Earth Engine (ee) downloads

In [8]:
initialize_ee()

# aoi: NC boundary -> ee.Geometry (GEE wants lat/long/EPSG:4326)
aoi = mask_to_ee_geometry(nc)

ee download of GABAM GeoTIFF images clipped to NC

In [ ]:
# GABAM collection
gabam = ee.ImageCollection("projects/sat-io/open-datasets/GABAM")

out_dir = data_path('raw', 'fire', 'gabam')
os.makedirs(out_dir, exist_ok=True)

years = range(1985, 2022)

for y in years:
    col = (gabam
           .filter(ee.Filter.stringEndsWith('system:index', f'_{y}'))
           .filterBounds(aoi))
    n = col.size().getInfo()
    
    if n == 0:
        print(f'{y}: no tiles for this year, skipping')   # e.g. 1986, 1988, 1990, 1991... aren't in GABAM
        continue
    
    img = col.mosaic().clip(aoi)
    out = os.path.join(out_dir, f'gabam_{y}_nc.tif')
    print(f'{y}: {n} tile(s) -> {out}')
    geemap.download_ee_image(
        image=img,
        filename=out,
        region=aoi,
        scale=30,
        crs='EPSG:4326',
        dtype='uint8',
    )

1985: 2 tile(s) -> /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/raw/fire/gabam/gabam_1985_nc.tif


/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/geemap/common.py:12263: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/180 tiles [00:00<?]

1986: no tiles for this year, skipping
1987: 2 tile(s) -> /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/raw/fire/gabam/gabam_1987_nc.tif


  0%|          |0/180 tiles [00:00<?]

...

  0%|          |0/180 tiles [00:00<?]

2020: 2 tile(s) -> /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/raw/fire/gabam/gabam_2020_nc.tif


  0%|          |0/180 tiles [00:00<?]

2021: 2 tile(s) -> /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/raw/fire/gabam/gabam_2021_nc.tif


  0%|          |0/180 tiles [00:00<?]

ee download of FireCCI51 images clipped to nc

In [ ]:
firecci51 = ee.ImageCollection('ESA/CCI/FireCCI/5_1').filterDate(
    '2001-01-01', '2020-12-31'
)

col = (firecci51.filterBounds(aoi))

out_dir = data_path('raw', 'fire', 'firecci51')
os.makedirs(out_dir, exist_ok=True)

n = col.size().getInfo()
print(f'{n} monthly images to download')

img_list = col.toList(n)
dates = col.aggregate_array('system:time_start').getInfo() # ms epoch per image

for i, t in enumerate(dates):
    ym = datetime.fromtimestamp(t / 1000, tz=timezone.utc).strftime('%Y_%m')
    img = ee.Image(img_list.get(i)).clip(aoi)
    
    out = os.path.join(out_dir, f'firecci51_{ym}_nc.tif')
    
    print(f'{ym}: -> {out}')
    
    geemap.download_ee_image(
        image=img,
        filename=out,
        region=aoi,
        scale=250, # FireCCI 5.1 native resolution is ~250 m
        crs='EPSG:4326',
        dtype='int16' # BurnDate is 0-366, ConfidenceLevel 0-100
    )

240 monthly images to download
2001_01: -> /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/raw/fire/firecci51/firecci51_2001_01_nc.tif


/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/geemap/common.py:12263: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


2001_01_01:   0%|          |0/12 tiles [00:00<?]

2001_02: -> /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/raw/fire/firecci51/firecci51_2001_02_nc.tif


2001_02_01:   0%|          |0/12 tiles [00:00<?]

...

2020_10_01:   0%|          |0/12 tiles [00:00<?]

2020_11: -> /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/raw/fire/firecci51/firecci51_2020_11_nc.tif


2020_11_01:   0%|          |0/12 tiles [00:00<?]

2020_12: -> /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/raw/fire/firecci51/firecci51_2020_12_nc.tif


2020_12_01:   0%|          |0/12 tiles [00:00<?]